In [ ]:
import os
import json
import random

from PIL import Image

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
MML_PATH = "path/to/data"

## Dataset overview

In [ ]:
mmlandmarks = pd.read_csv(os.path.join(MML_PATH,"mmlandmarks.csv"))
mmlandmarks.head()

In [ ]:
mml_train = pd.read_csv(os.path.join(MML_PATH, "train", "mml_train.csv"))
mml_query = pd.read_csv(os.path.join(MML_PATH, "query", "mml_query.csv"))

### Let's choose a landmark and determine which directory it is from (train or query):

In [ ]:
IDX = 1234

chosen_landmark = int(mmlandmarks["landmark_id"].iloc[IDX])
print(f"{chosen_landmark=}")

folder = "train" if chosen_landmark in list(mml_train["landmark_id"]) else "query"
print(f"{folder=}")

## Visualization

In [ ]:
mml = mml_train if folder == "train" else mml_query

In [ ]:
# Information about the landmark id and location of the corresponding ground image files
mml_ground = pd.read_csv(os.path.join(MML_PATH, folder, f"mml_{folder}_ground.csv"))
mml_ground.head()

In [ ]:
# Information about the landmark id and location of the corresponding satellite image files
mml_satellite = pd.read_csv(os.path.join(MML_PATH, folder, f"mml_{folder}_satellite.csv"))
mml_satellite.head()

In [ ]:
# Information about the landmark id and location of the corresponding json text files
mml_text = pd.read_csv(os.path.join(MML_PATH, folder, f"mml_{folder}_text.csv"))
mml_text.head()

In [ ]:
# Information about the GPS coordinates
mml.head()

Given our landmark of interest, we use the `landmark_id` to find the location of the ground images, satellite images, and text information.</br>
We used the first landmark from the train set, the Kent Free Library, as an example.

There are four modalities associated with the landmarks:
- Ground: "b6e40405d61d4658 0ebf9ff497cc475a 95db11a10bd3..." - from `mml_train_ground.csv`
- Satellite: "c8ea40a5178db5c3 112d01d5b11f841e a1249a655c8e..." - from `mml_train_satellite.csv`
- Text: "5a9087ecc3602a48" - from `mml_train_text.csv`
- GPS: (41.153332,-81.361273) - from `mml_train.csv`

For ground, satellite and text, the names and folders follow the structure:</br>

`${a}/${b}/${c}/${id}.ext`

where `ext` depends on the modality (`{ground: '.jpg', satellite: '.png', text: '.json'}`) and `${a}`,`${b}`,`${c}` are the first 3 characters of `${id}`. </br>
As an example, the ground image `0123456789abcdef` will be stored in 

`0/1/2/0123456789abcdef.jpg`



In [ ]:
# Ground images

ground_files = mml_ground[mml_ground["landmark_id"]==chosen_landmark]
ground_images = []

for file in ground_files["images"].item().split(" "):
    
    path = os.path.join(MML_PATH, f"{folder}", "ground", file[0], file[1], file[2], f"{file}.jpg")
    
    img = Image.open(path)
    ground_images.append(img)
    
print(f"There are {len(ground_images)} ground images associated with this landmark")

In [ ]:
# Satellite images

satellite_files = mml_satellite[mml_satellite["landmark_id"]==chosen_landmark]
satellite_images = []

for file in satellite_files["images"].item().split(" "):
    
    path = os.path.join(MML_PATH, f"{folder}", "satellite", file[0], file[1], file[2], f"{file}.png")
    
    img = Image.open(path)
    satellite_images.append(img)
    
print(f"There are {len(satellite_images)} satellite images associated with this landmark")

In [ ]:
# Text information

text_file = mml_text[mml_text["landmark_id"]==chosen_landmark]

file = text_file["json"].item()

path = os.path.join(MML_PATH, f"{folder}", "text", file[0], file[1], file[2], f"{file}.json")

with open(path, "r") as f:
    text = json.load(f)

print(json.dumps(text, indent=4))

## Wrapping it all together:

In [ ]:
meta = mmlandmarks[mmlandmarks["landmark_id"] == chosen_landmark].iloc[0]

In [ ]:
n = 5
sample_ground = random.sample(ground_images, min(n, len(ground_images)))
sample_sat    = random.sample(satellite_images, min(n, len(satellite_images)))

fig, axes = plt.subplots(3, n, figsize=(15, 9))
fig.suptitle(
    f"{meta['CommonsCategory']}  |  GPS: {meta['lat']:.5f}, {meta['lon']:.5f}"
    f"  |  {meta['hierarchical_category']}, {meta['state']}",
    fontsize=11
)

for i in range(n):
    for row, imgs, label in [(0, sample_ground, "Ground"), (1, sample_sat, "Satellite")]:
        ax = axes[row, i]
        if i < len(imgs):
            ax.imshow(imgs[i])
        else:
            ax.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_ylabel(label, fontsize=10)

# Text row
for ax in axes[2]:
    ax.set_visible(False)
plt.show()

print(f"- {len(ground_images)} ground images.")
print(f"- {len(satellite_images)} satellite images.")
print(f"- GPS location: {meta['lat']:.5f}, {meta['lon']:.5f}")
print("- Text information:")
print(json.dumps(text,indent=4))

print(45*"=")
print("Useful additional information:")
print(f"Wikimedia Commons Link: https://commons.wikimedia.org/wiki/Category:{meta['CommonsCategory'].replace(' ','_')}")
print(f"Wikipedia Link: https://en.wikipedia.org/wiki/{meta['WikipediaPage'].replace(' ','_')}")
print(f"OSM information Link: https://www.openstreetmap.org/{meta['osm_type']}/{meta['osm_id']}")